# SBOM-to-Audit — Stage 2 Colab Checkpoint

This notebook independently verifies the **Ghost-Logger real-format vertical slice** from a clean GitHub checkout and an isolated Python environment.

It does not edit or push repository files. It verifies:

- strict validation of all committed source artefacts;
- package installation and dependency integrity;
- static quality and the complete regression suite;
- deterministic generation of six replay outputs;
- EvidencePack Schema v0.2 validation;
- the five-event Ghost-Logger trajectory;
- explicit human authorization and deadline completion;
- generation of pilot figures and tables from machine-readable outputs.


In [ ]:
# Change REF to a Stage 2 tag after one is created.
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"

print("Repository:", REPO_URL)
print("Reference:", REF)

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/sbom_to_audit")
VENV_DIR = Path("/content/sbom_to_audit_stage2_venv")
for path in (WORKDIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
# Create an isolated environment so Colab's preinstalled packages cannot affect the result.
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV_DIR)], check=True)

PYTHON = VENV_DIR / "bin" / "python"
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"],
    check=True,
)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")

In [ ]:
# Run the canonical Stage 2 release gate.
import json

RELEASE_REPORT = Path("/content/stage2_colab_release_validation.json")
subprocess.run(
    [str(PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)],
    check=True,
)

release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
assert release["status"] == "PASS", release["errors"]
assert len(release["deterministic_hashes"]) == 6
print("PASS: canonical release gate")
print(json.dumps(release["deterministic_hashes"], indent=2))

In [ ]:
# Generate a separate checkpoint run and its pilot paper assets.
OUTPUT_ROOT = Path("/content/stage2_checkpoint_outputs")
ASSET_ROOT = Path("/content/stage2_checkpoint_paper_assets")
for path in (OUTPUT_ROOT, ASSET_ROOT):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(
    [
        str(PYTHON),
        "-m",
        "sbom_to_audit.cli",
        "--scenario",
        "data/scenarios/ghost_logger.yaml",
        "--output-root",
        str(OUTPUT_ROOT),
    ],
    check=True,
)
subprocess.run(
    [
        str(PYTHON),
        "paper_assets/scripts/build_stage2_assets.py",
        "--output-root",
        str(OUTPUT_ROOT),
        "--destination",
        str(ASSET_ROOT),
    ],
    check=True,
)

In [ ]:
# Verify Stage 2 source, state, authorization, schema and metric expectations.
import csv
import json

from jsonschema import Draft202012Validator

pack = json.loads((OUTPUT_ROOT / "evidence_packs/ghost_logger.json").read_text())
metrics = json.loads((OUTPUT_ROOT / "metrics/ghost_logger_metrics.json").read_text())
source_manifest = json.loads(
    (OUTPUT_ROOT / "source_manifests/ghost_logger_sources.json").read_text()
)
schema = json.loads(Path("schemas/evidencepack_v0.2.schema.json").read_text())
errors = list(Draft202012Validator(schema).iter_errors(pack))
assert not errors, [error.message for error in errors]

with (OUTPUT_ROOT / "state_logs/ghost_logger.csv").open(newline="") as handle:
    states = list(csv.DictReader(handle))

assert len(source_manifest["sources"]) == 15
assert len(states) == 5
assert [row["observed_state"] for row in states] == [
    "Document No-Report",
    "Escalate",
    "Report-Ready",
    "Report-Ready",
    "Report-Ready",
]
assert pack["schema_version"] == "0.2"
assert pack["decision_state"]["recommended_state"] == "Report-Ready"
assert pack["decision_state"]["authorized_state"] == "Report"
assert metrics["EC"] == 1.0
assert metrics["TR"] == 1.0
assert metrics["CD"] == 1.0
assert metrics["AR"] == 1.0
assert metrics["SC"] == 1.0
assert metrics["EPG"] == 1
assert metrics["supplemental"]["source_integrity_ratio"] == 1.0

print("PASS: 15 verified sources")
print("PASS: five-event temporal trajectory")
print("PASS: EvidencePack v0.2 schema")
print("PASS: authorization and source integrity")
print(json.dumps(metrics, indent=2))

In [ ]:
# Display the two pilot vector figures in Colab.
from IPython.display import SVG, display

display(SVG(filename=str(ASSET_ROOT / "figures/prototype_architecture.svg")))
display(SVG(filename=str(ASSET_ROOT / "figures/ghost_logger_trajectory.svg")))

In [ ]:
# Preserve the checkpoint as a downloadable evidence bundle.
import datetime as dt
import hashlib
import json
import zipfile

environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "commit": COMMIT,
    "kernel_python": sys.version,
    "isolated_python": subprocess.check_output([str(PYTHON), "--version"], text=True).strip(),
    "platform": platform.platform(),
    "release_status": release["status"],
}
ENV_PATH = Path("/content/stage2_colab_environment.json")
ENV_PATH.write_text(json.dumps(environment, indent=2) + "\n")

BUNDLE = Path("/content/stage2_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RELEASE_REPORT, RELEASE_REPORT.name)
    archive.write(ENV_PATH, ENV_PATH.name)
    for root in (OUTPUT_ROOT, ASSET_ROOT):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, f"{root.name}/{path.relative_to(root)}")

digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("SHA-256:", digest)
print("Download the bundle from Colab's Files panel.")